# Module 6.6 — Taiwan Deployment Hands-On

**Module 6: Vision + Deployment | Week 2 | Lab (Session 6.6)**

*Source: New content (Session 31, 2026-05-18). Pairs with 6.5 — same architecture, your hands on it, the Spearman 0.927 headline reproduced on your machine. Pattern: M05-lab3 fill-in.*

---

In 6.5 you saw the multi-task PF head produce Spearman 0.927 against published Taiwan city rates — the best result among four candidates we tested. Today you reproduce that finding on bundled data, then explore the per-shelter pattern and the failure modes of the other candidates.

By the end of this lab you'll have:

- Reproduced the PF→AAC direct-transfer failure (sanity check: AUC ~0.595 on dogs)
- Trained the multi-task model and verified the trunk-LR finding (+0.066 over the PF-only baseline)
- Scored Taiwan COA via both the multi-task PF head and the PF-only baseline; computed aggregate metrics and per-shelter Spearman
- Built the 7-shelter table showing where each candidate works and where it fails
- (Stretch) Replicated the four-model comparison from the lecture, including the AAC-only collapse

You won't write the multi-task class — it's given (the SHARED-only variant from 6.5, which is the Taiwan-deployable shape). You'll write the training calls, the Taiwan-scoring code, and the per-shelter analysis.

In [ ]:
import os
import sys
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

## Setup — two bundles

Same two bundles as the lecture: `petfinder-mini.zip` (PetFinder + COA_OpenData.csv + the M05 toolkit) + `aac-mini.zip` (AAC subset + `aac_prep.py` loader). No photo bundle today — the Taiwan deployment finding is tabular-only. Replace the two URLs with the ones your instructor gave you.

In [ ]:
PF_URL  = 'CONFIGURE_ME'     # ← petfinder-mini URL
AAC_URL = 'CONFIGURE_ME'     # ← aac-mini URL
PF_DIR  = '/content/petfinder'
AAC_DIR = '/content/aac'

def fetch(url, target_dir, marker_file, label):
    if os.path.exists(f'{target_dir}/{marker_file}'):
        return
    if url == 'CONFIGURE_ME':
        raise RuntimeError(f'Set {label} above to your instructor URL.')
    os.makedirs(target_dir, exist_ok=True)
    import gdown
    zip_path = f'{target_dir}/_bundle.zip'
    gdown.download(url=url, output=zip_path, quiet=False, fuzzy=True)
    if not os.path.exists(zip_path):
        raise RuntimeError(f'{label} download failed — check URL and Drive share settings.')
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(target_dir)
    os.remove(zip_path)
    if not os.path.exists(f'{target_dir}/{marker_file}'):
        raise RuntimeError(f'{marker_file} not found after extraction — wrong zip?')

fetch(PF_URL,  PF_DIR,  'train.csv',   'PF_URL')
fetch(AAC_URL, AAC_DIR, 'aac_prep.py', 'AAC_URL')

sys.path.insert(0, AAC_DIR)
sys.path.insert(0, PF_DIR)
from aac_prep import load_aac

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Bundles ready; device={device}')


In [ ]:
# --- Data load (same prep as the lecture) ---
pf = pd.read_csv(f'{PF_DIR}/train.csv')
pf['adopted_in_30_days'] = (pf['AdoptionSpeed'] <= 2).astype(int)
pf['age_adult']      = (pf['Age'] >= 12).astype(int)
pf['is_young']       = (pf['Age'] < 6).astype(int)
pf['is_mixed_breed'] = ((pf['Breed1'] == 307) | (pf['Breed2'] != 0)).astype(int)
pf['gender_male']    = (pf['Gender'] == 1).astype(int)
pf['gender_female']  = (pf['Gender'] == 2).astype(int)
pf['sterilized_yes'] = (pf['Sterilized'] == 1).astype(int)

aac = load_aac()
T_AAC = {1: 7.21, 2: 25.82}
fast = np.zeros(len(aac), dtype=int)
for sid, T in T_AAC.items():
    m = (aac['Type'] == sid)
    fast[m & (aac['outcome_type'] == 'Adoption') & (aac['days_in_shelter'] <= T)] = 1
aac['adopted_fast_AAC'] = fast

coa_raw = pd.read_csv(f'{PF_DIR}/COA_OpenData.csv')

def encode_coa(df):
    out = pd.DataFrame()
    out['Type']           = df['animal_kind'].map({'狗': 1, '貓': 2})
    out['age_adult']      = df['animal_age'].isin(['ADULT', 'SENIOR']).astype(int)
    out['is_young']       = (df['animal_age'] == 'CHILD').astype(int)
    out['is_mixed_breed'] = df['animal_Variety'].fillna('').str.contains('混種').astype(int)
    out['gender_male']    = (df['animal_sex'] == 'M').astype(int)
    out['gender_female']  = (df['animal_sex'] == 'F').astype(int)
    out['sterilized_yes'] = (df['animal_sterilization'] == 'T').astype(int)
    out['shelter_name']   = df['shelter_name']
    return out

coa = encode_coa(coa_raw).dropna(subset=['Type'])
coa['Type'] = coa['Type'].astype(int)

print(f'PetFinder dogs: {(pf["Type"]==1).sum():,}  |  cats: {(pf["Type"]==2).sum():,}')
print(f'AAC       dogs: {(aac["Type"]==1).sum():,}  |  cats: {(aac["Type"]==2).sum():,}')
print(f'Taiwan COA dogs: {(coa["Type"]==1).sum():,} |  cats: {(coa["Type"]==2).sum():,}')

## Given — models and training helpers

Same `MultiDatasetMLP` (SHARED-only variant) and helpers from 6.5. The class only consumes SHARED features — no PF-specific or AAC-specific feature block — because Taiwan COA can't supply those at deployment time. `train_multitask`, `train_pf_only`, and `prep_species` are given; your work is in the task cells.

In [ ]:
from m05_labbench import _make_layers

SHARED = ['age_adult', 'is_young', 'is_mixed_breed', 'gender_male', 'gender_female', 'sterilized_yes']

class MultiDatasetMLP(nn.Module):
    """Shared trunk + per-dataset heads, SHARED-only inputs (Taiwan-deployable shape)."""

    def __init__(self, n_shared, shared_hidden=(64, 32), head_hidden=(16,), dropout=0.2):
        super().__init__()
        trunk_layers, trunk_out = _make_layers(n_shared, shared_hidden, dropout)
        self.trunk = nn.Sequential(*trunk_layers)
        self.trunk_dim = trunk_out
        pf_layers, pf_h = _make_layers(trunk_out, head_hidden, dropout)
        pf_layers.append(nn.Linear(pf_h, 1))
        self.head_pf = nn.Sequential(*pf_layers)
        aac_layers, aac_h = _make_layers(trunk_out, head_hidden, dropout)
        aac_layers.append(nn.Linear(aac_h, 1))
        self.head_aac = nn.Sequential(*aac_layers)

    def forward_pf(self, x_shared):
        return self.head_pf(self.trunk(x_shared)).squeeze(-1)
    def forward_aac(self, x_shared):
        return self.head_aac(self.trunk(x_shared)).squeeze(-1)
    def trunk_only(self, x_shared):
        return self.trunk(x_shared)


def to_t(arr):
    return torch.tensor(arr, dtype=torch.float32, device=device)


def prep_species(species_id):
    pf_s = pf[pf['Type'] == species_id].copy()
    aac_s = aac[aac['Type'] == species_id].copy()
    X_pf, X_aac = pf_s[SHARED].values.astype(np.float32), aac_s[SHARED].values.astype(np.float32)
    y_pf, y_aac = pf_s['adopted_in_30_days'].values, aac_s['adopted_fast_AAC'].values
    pf_tr, pf_te = train_test_split(np.arange(len(y_pf)),  test_size=0.25, random_state=42, stratify=y_pf)
    aac_tr, aac_te = train_test_split(np.arange(len(y_aac)), test_size=0.25, random_state=42, stratify=y_aac)
    scaler = StandardScaler().fit(np.vstack([X_pf[pf_tr], X_aac[aac_tr]]))
    return {
        'X_pf_tr':  scaler.transform(X_pf[pf_tr]).astype(np.float32),
        'X_pf_te':  scaler.transform(X_pf[pf_te]).astype(np.float32),
        'y_pf_tr':  y_pf[pf_tr], 'y_pf_te': y_pf[pf_te],
        'X_aac_tr': scaler.transform(X_aac[aac_tr]).astype(np.float32),
        'X_aac_te': scaler.transform(X_aac[aac_te]).astype(np.float32),
        'y_aac_tr': y_aac[aac_tr], 'y_aac_te': y_aac[aac_te],
        'scaler': scaler,
    }


def train_multitask(data, seed, alpha=0.5, epochs=100):
    torch.manual_seed(seed)
    model = MultiDatasetMLP(n_shared=len(SHARED)).to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3); loss_fn = nn.BCEWithLogitsLoss()
    Xpf_tr, Xaac_tr = to_t(data['X_pf_tr']), to_t(data['X_aac_tr'])
    ypf_tr, yaac_tr = to_t(data['y_pf_tr']), to_t(data['y_aac_tr'])
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss = alpha * loss_fn(model.forward_pf(Xpf_tr), ypf_tr) + \
               alpha * loss_fn(model.forward_aac(Xaac_tr), yaac_tr)
        loss.backward(); opt.step()
    return model


def train_pf_only(data, seed, epochs=100):
    torch.manual_seed(seed)
    n = len(SHARED)
    layers, prev = [], n
    for h in (64, 32):
        layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(0.2)]; prev = h
    layers.append(nn.Linear(prev, 1))
    model = nn.Sequential(*layers).to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3); loss_fn = nn.BCEWithLogitsLoss()
    Xpf_tr, ypf_tr = to_t(data['X_pf_tr']), to_t(data['y_pf_tr'])
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss_fn(model(Xpf_tr).squeeze(-1), ypf_tr).backward(); opt.step()
    return model


def train_aac_only(data, seed, epochs=100):
    torch.manual_seed(seed)
    n = len(SHARED)
    layers, prev = [], n
    for h in (64, 32):
        layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(0.2)]; prev = h
    layers.append(nn.Linear(prev, 1))
    model = nn.Sequential(*layers).to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3); loss_fn = nn.BCEWithLogitsLoss()
    Xaac_tr, yaac_tr = to_t(data['X_aac_tr']), to_t(data['y_aac_tr'])
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss_fn(model(Xaac_tr).squeeze(-1), yaac_tr).backward(); opt.step()
    return model


SEEDS = [42, 7, 2026]
data_dogs = prep_species(1)
print(f'Train sizes: PF dogs {len(data_dogs["y_pf_tr"]):,}  AAC dogs {len(data_dogs["y_aac_tr"]):,}')
print(f'Test sizes:  PF dogs {len(data_dogs["y_pf_te"]):,}  AAC dogs {len(data_dogs["y_aac_te"]):,}')

---

### Task 1: Reproduce the PF→AAC direct-transfer failure

First confirm the cross-dataset gap is real on your hardware. Train a `train_pf_only(data_dogs, seed)` model for each seed in `SEEDS`. For each, compute:

- **PF→PF same-dataset AUC** on `data_dogs['X_pf_te']` / `y_pf_te` (sanity check, should land near 0.67)
- **PF→AAC cross-dataset AUC** on `data_dogs['X_aac_te']` / `y_aac_te` (should land near 0.595 — the PF-only baseline)

Print 3-seed means for both. The gap between them — about 0.075 AUC — is what the rest of the lab tries to close.

In [ ]:
# Task 1: your code here


---

### Task 2: Train multi-task + verify trunk-LR

Train `train_multitask(data_dogs, seed)` for each seed, save the 3 models. For each, run the trunk-only linear probe — the M05-lab4 Task 3 recipe, now reproduced with the SHARED-only multi-task model:

1. Pass `data_dogs['X_aac_tr']` and `X_aac_te` through `model.trunk_only()` to get 32-d trunk representations.
2. Fit `LogisticRegression(max_iter=1000)` on the trunk-tr representation against `y_aac_tr`.
3. Predict on trunk-te; compute AUC against `y_aac_te`.

Print 3-seed mean trunk-LR AUC. Should land near **0.661** — that's +0.066 over Task 1's PF→AAC direct-transfer baseline of 0.595. The shared trunk learned features useful for both datasets, not just PetFinder.

In [ ]:
# Task 2: your code here


---

### Task 3: Score the multi-task PF head on Taiwan COA

Now the headline. Encode Taiwan COA dogs into the SHARED feature space (`coa_dogs = coa[coa['Type'] == 1]`), apply the scaler from `data_dogs['scaler']`, and score each of your 3 multi-task models via `model.forward_pf(...)`. Average the 3 seeds' probabilities into one per-animal prediction.

Compute:

- Mean prediction across all COA dogs (should land near **0.365**; target inferred at 0.453)
- Median, IQR (75th − 25th percentile)
- Percent of predictions above 0.5

Print the same aggregate row for `train_pf_only` (PF-only baseline) for comparison. Multi-task should win on **mean** (closer to 0.453), **IQR** (wider — ~0.21 vs ~0.11), and `%>0.5` — three out of four aggregate metrics.

In [ ]:
# Task 3: your code here


---

### Task 4: Per-shelter Spearman — the deployment-validity test

The metric that matters most for shelter operations: does the model's per-shelter ordering match Taiwan's published per-city ordering?

Use this top-7 shelter list with published 2024 rates:

```python
TOP_SHELTERS = {
    '新北市八里區公立動物之家': 0.95,    # New Taipei, urban
    '新北市五股區公立動物之家': 0.95,
    '新北市新店區公立動物之家': 0.95,
    '臺北市動物之家':           0.88,    # Taipei City
    '高雄市燕巢動物保護關愛園區': 0.77,   # Kaohsiung
    '南投縣公立動物收容所':     0.47,    # Nantou, rural
    '雲林縣流浪動物收容所':     0.39,    # Yunlin, rural
}
```

For each shelter:

1. Subset `coa_dogs` to rows where `shelter_name` matches (some won't have any listings — skip those).
2. Compute the mean PF-only prediction and the mean multi-task PF head prediction over those listings.
3. Build a small DataFrame with shelter / published_rate / pf_pred / mt_pred / n_listings.

Compute `spearmanr(published_rate, pf_pred).statistic` and `spearmanr(published_rate, mt_pred).statistic`. Multi-task should land near **0.927** — the lecture's headline finding, reproduced on your hardware. PF-only should land near 0.778.

Hint: `from scipy.stats import spearmanr`.

In [ ]:
# Task 4: your code here


---

### Task 5 (stretch): Four-model comparison

Replicate the full four-model Taiwan comparison from the lecture by also training `train_aac_only(data_dogs, seed)`. Then score Taiwan COA via four candidates:

- PF-only (baseline)
- AAC-only (the collapse case)
- Multi-task PF head (the deployment recipe)
- Multi-task AAC head (for comparison — also expected to under-predict)

Build the same aggregate table as Task 3 with all four. The AAC-only mean should land near **0.172** — much lower than the inferred target of 0.453 — because AAC's signal lives in features Taiwan can't supply, and on SHARED features alone the AAC head essentially predicts the AAC base rate.

Write down: which model would you ship to a shelter coordinator, and what's the one-line caveat that goes with it?

In [ ]:
# Task 5 (stretch): your code here


---

## Writeup

In a markdown cell at the bottom, write **5–7 bullets** answering:

- (Task 1) PF→PF AUC and PF→AAC AUC — what's the gap? Did the PF-only baseline (0.595) reproduce on your hardware?
- (Task 2) Trunk-LR AUC on AAC — did it clear the PF-only baseline by +0.05+ (the "real signal" threshold)?
- (Task 3) Multi-task PF head's Taiwan aggregate — mean, IQR, gap vs target. How does it compare to PF-only?
- (Task 4) Spearman ρ for PF-only and for multi-task — did multi-task land near 0.927?
- (Task 5, optional) Did AAC-only collapse on Taiwan? What does that tell you about AAC's signal source?
- One sentence: in your own words, **what does Spearman 0.93 mean operationally** — what kind of decisions does the multi-task PF head deployment support?

---

## Summary

- Reproduced the PF→AAC direct-transfer failure (PF-only baseline ~0.595 on dogs).
- Verified the trunk-LR finding (+0.066 over PF-only — the shared trunk transfers).
- Scored the multi-task PF head on Taiwan COA; computed aggregate metrics + the per-shelter Spearman.
- Built the 7-shelter deployment table — saw urban vs rural pattern preserved at Spearman ~0.93.
- (Stretch) Reproduced the AAC-only collapse on Taiwan — different shelters, different signal sources.

You now have a working Taiwan deployment recipe and the empirical evidence that justifies it. **Next**: 6.7 makes the ethics and limitations explicit, and 6.9 ships the deployment as a written brief.

---

**Next session (Lecture 6.7):** Responsible vision-ML — ranking-only validity in plain language, the M07 handoff.